In [ ]:
!pip install transformers datasets torch scikit-learn pandas accelerate

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel, pipeline
import kagglehub
import pandas as pd

# Set device (GPU if available)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# base BERT model
MODEL_NAME = "distilbert-base-uncased"

# Load tokenizer and base model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Enable hidden states output
bert_model = AutoModel.from_pretrained(
    MODEL_NAME,
    output_hidden_states=True
)

# Set evaluation mode (no dropout)
bert_model.eval()

# Move model to device
bert_model.to(device)

# Display basic model info
print(f"Model: {MODEL_NAME}")
print(f"Layers: {bert_model.config.num_hidden_layers}")
print(f"Hidden size: {bert_model.config.hidden_size}")

Using device: cuda


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model: distilbert-base-uncased
Layers: 6
Hidden size: 768


In [ ]:
sentence1 = "The river bank was muddy after the rain."
sentence2 = "I went to the bank to deposit money."

In [ ]:
def get_token_embeddings(sentence, model, tokenizer, device):
    """
    Tokenize input sentence and extract last-layer embeddings
    """

    # Tokenize input
    inputs = tokenizer(sentence, return_tensors='pt')

    # Move to device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Forward pass (no gradients needed)
    with torch.no_grad():
        outputs = model(**inputs)

    # Extract last hidden layer (contextual embeddings)
    last_hidden = outputs.hidden_states[-1][0]  # (seq_len, hidden_size)

    # Convert IDs to readable tokens
    tokens = tokenizer.convert_ids_to_tokens(
        inputs['input_ids'][0].tolist()
    )

    return tokens, last_hidden.cpu()

In [ ]:
tokens1, embeddings1 = get_token_embeddings(sentence1, bert_model, tokenizer, device)
tokens2, embeddings2 = get_token_embeddings(sentence2, bert_model, tokenizer, device)

print("Tokens Sentence 1:", tokens1)
print("Tokens Sentence 2:", tokens2)

Tokens Sentence 1: ['[CLS]', 'the', 'river', 'bank', 'was', 'muddy', 'after', 'the', 'rain', '.', '[SEP]']
Tokens Sentence 2: ['[CLS]', 'i', 'went', 'to', 'the', 'bank', 'to', 'deposit', 'money', '.', '[SEP]']


In [ ]:
def find_token_index(tokens, target_word):
    """
    Find token index (handles subwords like '##ing')
    """
    for i, tok in enumerate(tokens):
        if tok == target_word or tok.replace("##", "") == target_word:
            return i
    return None

target_word = "bank"

idx1 = find_token_index(tokens1, target_word)
idx2 = find_token_index(tokens2, target_word)

print(f'"bank" index in sentence 1: {idx1}: {tokens1[idx1]}')
print(f'"bank" index in sentence 2: {idx2}: {tokens2[idx2]}')

"bank" index in sentence 1: 3: bank
"bank" index in sentence 2: 5: bank


In [ ]:
# Extract vectors (768 dimensions)
bank_emb1 = embeddings1[idx1]
bank_emb2 = embeddings2[idx2]

print("Embedding shape:", bank_emb1.shape)

Embedding shape: torch.Size([768])


In [ ]:
# Compute cosine similarity
cosine_sim = F.cosine_similarity(
    bank_emb1.unsqueeze(0),
    bank_emb2.unsqueeze(0)
).item()

print(f"\nCosine similarity: {cosine_sim:.4f}")


Cosine similarity: 0.6823


In [ ]:
print("\nInterpretation:")

if cosine_sim > 0.9:
    print("Very similar: model treats both contexts similarly")
elif cosine_sim > 0.7:
    print("Moderately similar: some contextual difference")
else:
    print("Low similarity: strong contextual distinction")


Interpretation:
Low similarity: strong contextual distinction


Discussion

Contextual embeddings

BERT assigns different vectors to "bank" depending on context

River bank, financial bank: different meanings: different vectors

Static embeddings (Word2Vec/GloVe):

Only ONE vector per word

cosine similarity = 1

In [ ]:
# Pre-trained sentiment classifier
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=0 if torch.cuda.is_available() else -1
)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [ ]:
test_sentences = [
    "This movie was absolutely amazing!",
    "I hated every minute of this film.",
    "It was okay, nothing special.",
    "One of the worst movies I’ve seen."
]

In [ ]:
results = sentiment_pipeline(test_sentences)

print("\nSentiment Analysis Results:")
print(f"{'Sentence':<50} {'Label':<10} {'Score'}")
print("-" * 75)

for sent, res in zip(test_sentences, results):
    print(f"{sent:<50} {res['label']:<10} {res['score']:.4f}")


Sentiment Analysis Results:
Sentence                                           Label      Score
---------------------------------------------------------------------------
This movie was absolutely amazing!                 POSITIVE   0.9999
I hated every minute of this film.                 NEGATIVE   0.9995
It was okay, nothing special.                      NEGATIVE   0.9821
One of the worst movies I’ve seen.                 NEGATIVE   0.9998


Discussion

Ease of pre-trained models using pipeline:

Only 2-3 lines of code, no training required, instant predictions, sota performance

Training from scratch:

Need data preprocessing, training, hyperparameter tuning, time-consuming evaluation

In [ ]:
# Load dataset
path = kagglehub.dataset_download("columbine/imdb-dataset-sentiment-analysis-in-csv-format")
print("Dataset path:", path)

Using Colab cache for faster access to the 'imdb-dataset-sentiment-analysis-in-csv-format' dataset.
Dataset path: /kaggle/input/imdb-dataset-sentiment-analysis-in-csv-format


In [ ]:
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import accuracy_score, f1_score

import os

# Find CSV file inside downloaded path
files = os.listdir(path)
print("Files in dataset folder:", files)

# .csv extension
csv_file = [f for f in files if f.endswith('.csv')][1]

df = pd.read_csv(os.path.join(path, csv_file), nrows=5000)

# Display dataset info
print("\nDataset Shape:", df.shape)
print("\nColumns:", df.columns)

# Show first 5 rows
print("\nFirst 5 rows:")
print(df.head())

# Create Hugging Face Dataset from existing DataFrame
hf_dataset = Dataset.from_pandas(df[['text', 'label']])

print(hf_dataset)
print(hf_dataset.features)

Files in dataset folder: ['Valid.csv', 'Train.csv', 'Test.csv']

Dataset Shape: (5000, 2)

Columns: Index(['text', 'label'], dtype='object')

First 5 rows:
                                                text  label
0  I grew up (b. 1965) watching and loving the Th...      0
1  When I put this movie in my DVD player, and sa...      0
2  Why do people who do not know what a particula...      0
3  Even though I have great interest in Biblical ...      0
4  Im a die hard Dads Army fan and nothing will e...      1
Dataset({
    features: ['text', 'label'],
    num_rows: 5000
})
{'text': Value('string'), 'label': Value('int64')}


In [ ]:
# Split into train/test
split = hf_dataset.train_test_split(test_size=0.2, seed=42)

train_ds = split['train']
test_ds  = split['test']

print(f"Train size: {len(train_ds)}")
print(f"Test size: {len(test_ds)}")

Train size: 4000
Test size: 1000


In [ ]:
MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
def tokenize_function(examples):
    """
    Convert raw text into model inputs
    - padding to fixed length
    - truncation for long reviews
    """
    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=128
    )

# Apply tokenization (batched is faster)
train_ds = train_ds.map(tokenize_function, batched=True)
test_ds  = test_ds.map(tokenize_function, batched=True)

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
# Keep only required columns for training
train_ds.set_format(
    type='torch',
    columns=['input_ids', 'attention_mask', 'label']
)

test_ds.set_format(
    type='torch',
    columns=['input_ids', 'attention_mask', 'label']
)

In [ ]:
# Load classification model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2   # binary sentiment
)

# Move to device
model.to(device)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [ ]:
training_args = TrainingArguments(
    output_dir="./results",            # where model checkpoints go
    num_train_epochs=2,                # small but effective for fine-tuning
    per_device_train_batch_size=16,    # balanced memory vs speed
    per_device_eval_batch_size=16,
    learning_rate=2e-5,                # standard for BERT fine-tuning
    weight_decay=0.01,                # regularization
    do_eval=True,
    logging_dir="./logs",
    logging_steps=50,
    save_steps=500
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
import numpy as np

def compute_metrics(eval_pred):
    """
    Convert logits: predictions: compute metrics
    """
    logits, labels = eval_pred

    # Convert logits to predicted class
    preds = np.argmax(logits, axis=1)

    acc = accuracy_score(labels, preds)
    f1  = f1_score(labels, preds)

    return {
        "accuracy": acc,
        "f1": f1
    }

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics
)

In [ ]:
pip uninstall -y torchvision

In [ ]:
import sys
print("torchvision" in sys.modules)   #should be False

False


In [ ]:
### restart session and run all ### to remove torchvision

print("Starting training...\n")

trainer.train()

Starting training...



Step,Training Loss
50,0.649791
100,0.430504
150,0.396615
200,0.375515
250,0.396855
300,0.264217
350,0.270951
400,0.246201
450,0.268001
500,0.276310


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=500, training_loss=0.3574959621429443, metrics={'train_runtime': 106.4322, 'train_samples_per_second': 75.165, 'train_steps_per_second': 4.698, 'total_flos': 264934797312000.0, 'train_loss': 0.3574959621429443, 'epoch': 2.0})

In [ ]:
print("\nEvaluating model...\n")

results = trainer.evaluate()

print("=== Transformer (DistilBERT) Results ===")
for k, v in results.items():
    print(f"{k}: {v:.4f}")


Evaluating model...



Training Loss,Validation Loss,Step,Accuracy,F1
0.276310,0.410647,500,0.840000,0.836066


=== Transformer (DistilBERT) Results ===
eval_loss: 0.4106
eval_accuracy: 0.8400
eval_f1: 0.8361


LSTM:

Accuracy: 0.7985

F1: 0.7970

Fine-Tuned DistilBERT:

Accuracy: 0.84

F1: 0.8361

Transformer outperforms LSTM

Discussion

Hyperparameter Choices

**2 epochs**

Fine-tuning converges quickly and more epochs has risk of overfitting

**Batch size 16**

Balances memory usage and training stability

**Learning rate 2e-5**

Standard for BERT

Too high destroys pre-trained weights, too low slow learning

**Weight decay 0.01**

reduce overfitting

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch

# Device setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

print('Loading GPT-2...')

MODEL_NAME = 'gpt2'

# Load tokenizer + model
gpt_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
gpt_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

gpt_model.eval()
gpt_model.to(device)

# GPT-2 has no padding token, used EOS token
gpt_tokenizer.pad_token = gpt_tokenizer.eos_token

print(f'GPT-2 parameters: {sum(p.numel() for p in gpt_model.parameters()):,}')

Device: cuda
Loading GPT-2...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT-2 parameters: 124,439,808


In [ ]:
def generate_text(prompt, model, tokenizer, device, max_new_tokens=80, **kwargs):
    """
    Generate text continuation from a prompt
    """

    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    input_len = inputs['input_ids'].shape[1]

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id,
            **kwargs
        )

    # Return only generated portion
    generated = tokenizer.decode(
        output_ids[0][input_len:],
        skip_special_tokens=True
    )

    return generated

In [ ]:
prompt1 = "Natural language processing is"
print(f'Prompt: "{prompt1}"\n')

Prompt: "Natural language processing is"



In [ ]:
greedy = generate_text(
    prompt1,
    gpt_model,
    gpt_tokenizer,
    device,
    do_sample=False
)

print("Greedy output:")
print(greedy)

Greedy output:
 a very important part of the language learning process.

The first step is to understand the language. The second step is to understand the language.

The first step is to understand the language.

The second step is to understand the language.

The third step is to understand the language.

The fourth step is to understand the language.

The fifth step is


In [ ]:
temperatures = [0.3, 0.7, 1.2]

for temp in temperatures:
    out = generate_text(
        prompt1,
        gpt_model,
        gpt_tokenizer,
        device,
        do_sample=True,
        temperature=temp,
        max_new_tokens=60
    )

    print(f"\nTemperature={temp}")
    print(out)



Temperature=0.3
 a powerful tool for the human brain. It allows us to understand the world around us and to understand our own thoughts and feelings.

The brain is a complex machine. It's a complex machine that can be programmed to do many things. We can do many things, but we can't do

Temperature=0.7
 essential to the development of language learning and is a critical component of any successful language learning.

In this article we will discuss how to use this technique in a variety of situations and explain how it is helpful to use it.

What is Language Learning?

Language learning is a process

Temperature=1.2
 not required - rather this way the user can use the feature easily with complex language learning tasks. In particular, when the user attempts to create complex rules from scratch the only input are the rules and parameters and a set of words. This feature has no effect on writing and creating rules based entirely on an


In [ ]:
configs = [
    {'do_sample': True, 'top_k': 10,  'temperature': 0.7},
    {'do_sample': True, 'top_k': 50,  'temperature': 0.7},
    {'do_sample': True, 'top_p': 0.9, 'temperature': 0.7},
    {'do_sample': True, 'top_p': 0.95,'temperature': 1.0},
]

for cfg in configs:
    label = ', '.join(f'{k}={v}' for k, v in cfg.items())

    out = generate_text(
        prompt1,
        gpt_model,
        gpt_tokenizer,
        device,
        max_new_tokens=60,
        **cfg
    )

    print(f"\n{label}")
    print(out)


do_sample=True, top_k=10, temperature=0.7
 not as straightforward as it should be, as the above examples demonstrate.

The second problem is that, even if the language processing is implemented on an object, it will be quite difficult to use a single language processing program.

One way to deal with this problem is to write your own

do_sample=True, top_k=50, temperature=0.7
 a significant source of computational power. It is also a source of new technologies.

The next step is to create a machine that can interpret human speech and respond to human emotions. This machine needs to be very smart and accurate – the person who interprets human speech says it is coming from inside

do_sample=True, top_p=0.9, temperature=0.7
 a common feature of computer languages, especially those that have a deep understanding of basic linguistic syntax. We can use the techniques we've learned to develop the language, and then use it to express our own concepts.

What are the pros and cons of using an a

In [ ]:
def generate_multiple(prompt, n=3):
    inputs = gpt_tokenizer(prompt, return_tensors='pt').to(device)
    input_len = inputs['input_ids'].shape[1]

    with torch.no_grad():
        outputs = gpt_model.generate(
            **inputs,
            max_new_tokens=60,
            num_return_sequences=n,
            do_sample=True,
            temperature=0.8,
            top_p=0.9,
            pad_token_id=gpt_tokenizer.eos_token_id
        )

    return [
        gpt_tokenizer.decode(out[input_len:], skip_special_tokens=True)
        for out in outputs
    ]

prompt2 = "The future of AI might be"

results = generate_multiple(prompt2, n=3)

print(f'Prompt: "{prompt2}"\n')

for i, text in enumerate(results, 1):
    print(f"[{i}] {text}\n")

Prompt: "The future of AI might be"

[1]  a lot brighter if it can be learned from other people.

[2]  much different than it was in the past, and this will be the story of the future of the AI that we have to see.

You are currently working on the game, and the game is currently in alpha. What are some of your thoughts on the future of AI?

We

[3]  on the line.

I am aware that a lot of the AI research in this space is focused on human-level control over the human brain, so I would like to take this opportunity to make this a little bit more concrete.

I'm going to try to make it clear that



Effect of Parameters

temperature lower, more focused/repetative. temp higher, more creative/random.

do sample false is greedy which was safe but repetative

do sample true with top k or top p: It looked like the higher top k and p are better but I think top p 0.9 is the balance we look for.

In [ ]:
article = (
    "Scientists developed a model that predicts Alzheimer's up to six years early "
    "by analyzing speech patterns. It achieves about 80% accuracy and could help "
    "early treatment decisions."
)

summary_prompt = f"""
Summarize this in one sentence:

{article}

Summary:
"""

summary = generate_text(
    summary_prompt,
    gpt_model,
    gpt_tokenizer,
    device,
    do_sample=False,
    max_new_tokens=50
)

print("Article:")
print(article)
print("\nGPT-2 Summary:")
print(summary)

Article:
Scientists developed a model that predicts Alzheimer's up to six years early by analyzing speech patterns. It achieves about 80% accuracy and could help early treatment decisions.

GPT-2 Summary:

The model predicts that Alzheimer's will be diagnosed by the end of the century.

The model predicts that Alzheimer's will be diagnosed by the end of the century.

The model predicts that Alzheimer's will be diagnosed by the end of


In [ ]:
context = (
    "The Eiffel Tower is in Paris. It was completed in 1889 "
    "and stands 330 meters tall."
)

question = "How tall is the Eiffel Tower?"

qa_prompt = f"""
Context: {context}

Question: {question}

Answer:
"""

answer = generate_text(
    qa_prompt,
    gpt_model,
    gpt_tokenizer,
    device,
    do_sample=False,
    max_new_tokens=30
)

print("\nGPT-2 Answer:")
print(answer)


GPT-2 Answer:

The Eiffel Tower is the tallest building in the world. It is the tallest building in the world. It is the tallest building in the


Although there are simple summaries and basic QA, simple prompting was not effective.

Limitation:

GPT-2 is not instruction-tuned and actually gave repetitions and ignored instructions
